In [ ]:
from mne.minimum_norm import make_inverse_operator, apply_inverse

import mne
import matplotlib.pyplot as plt
import numpy as np
from mne.stats import permutation_cluster_test
from mne.stats import ttest_ind_no_p
import pandas as pd

%matplotlib qt5
 
import os
import pandas as pd
from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
import mne
import PyQt5
from matplotlib.colors import TwoSlopeNorm
import seaborn as sns

from mne.stats import permutation_cluster_test as pcluster_test

from mne.datasets import sample
from mne.minimum_norm import apply_inverse, make_inverse_operator

import pickle

In [ ]:
path_to_ALL_ALL_onset_epoches = Path("/Users/alexmedvedev/Desktop/Vishka/ALL_ALL/onset")
path_to_cov = Path('/Users/alexmedvedev/Desktop/Vishka/Loreta/cov')
path_to_fwd = Path('/Users/alexmedvedev/Desktop/Vishka/Loreta/frw')
path_to_inv = Path('/Users/alexmedvedev/Desktop/Vishka/Loreta/inv')
path_to_loreta = Path('/Users/alexmedvedev/Desktop/Vishka/Loreta')
path_to_avg_loreta = Path('/Users/alexmedvedev/Desktop/Vishka/Loreta/loreta_avg')
path_to_cov_loreta = Path('/Users/alexmedvedev/Desktop/Vishka/Loreta/loreta_cov')
path_to_avg_cov_loreta = Path('/Users/alexmedvedev/Desktop/Vishka/Loreta/loreta_cov/avg')

In [ ]:
all_dis_2 = [file for file in path_to_ALL_ALL_onset_epoches.iterdir() if file.name.endswith('fif')]
all_dis_2

In [ ]:
all_dis_epochs_2 = {file.name: None for file in all_dis_2}

for file in all_dis_2: 
    dis_epochs_2 = mne.read_epochs(file)
    dis_epochs_2 = dis_epochs_2 ['5-6', '7-8', '9-10', '11-12']
    #dis_epochs_2 = dis_epochs_2['contr_9-10'] # 5-6 7-8 9-10 11-12 contr_5-6 contr_7-8 contr_9-10 contr_11-12
    all_dis_epochs_2[file.name] = dis_epochs_2

In [ ]:
all_dis_erps_2 = {name : {} for name in all_dis_epochs_2}
for name in all_dis_epochs_2:
    for event_id in all_dis_epochs_2[name].event_id:
        current_dis_epochs_2 = all_dis_epochs_2[name][event_id]
        dis_erp_2 = current_dis_epochs_2.copy().load_data().interpolate_bads().average()
        all_dis_erps_2[name][event_id] = dis_erp_2

## create cov

In [ ]:
all_cov = {name : {} for name in all_dis_epochs_2}
for name in all_dis_epochs_2:
    for event_id in all_dis_epochs_2[name].event_id:
        current_data = all_dis_epochs_2[name][event_id]
        noise_cov = mne.compute_covariance(
        current_data, tmax=0.0, method="auto", rank=None, verbose=True
)
        all_cov[name][event_id] = noise_cov

#fig_cov, fig_spectra = mne.viz.plot_cov(noise_cov, raw.info)

In [ ]:
# Ensure the save directory exists
path_to_cov.mkdir(parents=True, exist_ok=True)

for epoch_file in all_dis_2:
    epochs = mne.read_epochs(epoch_file, preload=True)
    
    # Compute covariance matrix
    cov = mne.compute_covariance(epochs, tmin=None, tmax=0, method='auto', rank=None, verbose=True)
    
    # Extract the base filename without the directory and extension
    base_filename = Path(epoch_file).stem  # This gets the filename without the directory and suffix

    # Save covariance matrix with custom naming
    save_path = path_to_cov / f'cov_{base_filename}.cov'  # Construct the full save path
    cov.save(save_path, overwrite=True)  # Save the covariance matrix


In [ ]:
all_cov

## EEG forward operator

In [ ]:
from mne.datasets import eegbci, fetch_fsaverage
import os.path as op
import nibabel

# Download fsaverage files
fs_dir = fetch_fsaverage(verbose=True)
subjects_dir = op.dirname(fs_dir)

# The files live in:
subject = "fsaverage"
trans = "fsaverage"  # MNE has a built-in fsaverage transformation
src = op.join(fs_dir, "bem", "fsaverage-ico-5-src.fif")
bem = op.join(fs_dir, "bem", "fsaverage-5120-5120-5120-bem-sol.fif")

In [ ]:
montage = mne.channels.make_standard_montage("standard_1020")
# Read and set the EEG electrode locations, which are already in fsaverage's
# space (MNI space) for standard_1020:
all_bad_channels = []
inv_list = {file.name : None for file in all_dis_2}
for file in all_dis_2:
    raw = mne.read_epochs(file, preload = True)
    common_bads = set(raw[0].info['bads'])
    all_bad_channels.append (common_bads)

# Step 1: Flatten the list of sets into a single set to remove duplicates
combined_channels = set().union(*all_bad_channels)

# Step 2: Convert the set to a list
channel_list = list(combined_channels)

# Print the final list of channel names
print(channel_list)


for file in all_dis_2:
    raw = mne.read_epochs(file, preload = True)
    raw.set_montage (montage)
    epochs_cleaned = raw.copy().drop_channels(channel_list)
    epochs_cleaned.set_eeg_reference(projection=True)  # needed for inverse modeling
    inv_list[file.name] = epochs_cleaned

In [ ]:
#fwd_list = []
fwd_list = {}
montage = mne.channels.make_standard_montage("standard_1020")
for file in all_dis_2:
    print (file)
    raw = mne.read_epochs(file, preload = True)
    raw.set_montage (montage)
    raw = raw.copy().drop_channels(channel_list)
    print(raw.info)
    fwd = mne.make_forward_solution(
        raw.info, trans=trans, src=src, bem=bem, eeg=True, mindist=5.0, n_jobs=None
)
    fwd_list[file] = fwd

In [ ]:
new_fwd_list = {}
for key, value in fwd_list.items():
    new_key = os.path.basename(str(key))
    new_fwd_list[new_key] = value

# Replace the original fwd_list with the new one
fwd_list = new_fwd_list
new_fwd_list

In [ ]:
for eeg_file, fwd in new_fwd_list.items():
    mne.write_forward_solution(Path('/Users/alexmedvedev/Desktop/Vishka/Loreta/frw')/f'{eeg_file}_fwd.fif', fwd, overwrite = True)

In [ ]:
fwd_names = [file for file in path_to_fwd.iterdir() if file.name.endswith('.fif')]
fwd_names

In [ ]:
fwd_operators = {}
for fwd_file in fwd_names:
    fwd = mne.read_forward_solution(fwd_file)
    fwd_operators[os.path.basename(fwd_file)] = fwd
    #fwd_operators[fwd_file] = fwd

In [ ]:
fwd_operators

## Inverse operator

In [ ]:
cov_names = [file for file in path_to_cov.iterdir() if file.name.endswith('.cov')]
cov_names

In [ ]:
fwd_names = [file for file in path_to_fwd.iterdir() if file.name.endswith('.fif')]
fwd_names

In [ ]:
cov_list = {}
for cov_file in cov_names:
    cov = mne.read_cov(path_to_cov / f'cov_{base_filename}.cov')
    cov_list[os.path.basename(cov_file)] = cov

In [ ]:
cov_list

In [ ]:
inverse_operators_2 = []

for epoch, fwd_file, cov_file in zip(all_dis_2, fwd_names, cov_names):
    data = mne.read_epochs(epoch, preload = True)
    data.set_eeg_reference(projection=True)
    fwd = mne.read_forward_solution(fwd_file)
    cov = mne.read_cov(path_to_cov / f'{cov_file}')
    
    inverse_operator = mne.minimum_norm.make_inverse_operator(
        data.info, fwd, cov, loose=0.2, depth=0.8)
    
    #print(inverse_operator)
    inverse_operators_2.append(inverse_operator)

In [ ]:
inverse_operators_2

In [ ]:
for inv_number, inv in enumerate(inverse_operators_2):
    mne.minimum_norm.write_inverse_operator(path_to_inv/f'{str(inv_number + 1).zfill(3)}_inv.fif', inv, overwrite = True)

## SNR

In [ ]:
all_dis_epochs_all = []
epochs_5 = []
epochs_7 = []
epochs_9 = []
epochs_11 = []

for file in all_dis_2: 
    dis_epochs = mne.read_epochs(file)
    dis_epochs_5 = dis_epochs ['5-6'].average()
    epochs_5.append(dis_epochs_5)
    dis_epochs_7 = dis_epochs ['7-8'].average()
    epochs_7.append(dis_epochs_7)
    dis_epochs_9 = dis_epochs ['9-10'].average()
    epochs_9.append(dis_epochs_9)
    dis_epochs_11 = dis_epochs ['11-12'].average()
    epochs_11.append(dis_epochs_11)
    #dis_epochs_2 = dis_epochs_2['contr_9-10'] # 5-6 7-8 9-10 11-12 contr_5-6 contr_7-8 contr_9-10 contr_11-12

grand_avg_5 = mne.grand_average(epochs_5)
grand_avg_7 = mne.grand_average(epochs_7)
grand_avg_9 = mne.grand_average(epochs_9)
grand_avg_11 = mne.grand_average(epochs_11)

all_dis_epochs_all = [grand_avg_5, grand_avg_7, grand_avg_9, grand_avg_11]

In [ ]:
names = ['5-6', '7-8', '9-10', '11-12']
counter = 0
for group in all_dis_epochs_all:
    group.set_eeg_reference(projection=True)

    print(inverse_operators_2)
    print(type(inverse_operators_2))
    # Calculate MNE:
    snr = 3.0
    lambda2 = 1.0 / snr ** 2
    stc = apply_inverse(group, inverse_operators_2[0], lambda2, 'MNE', verbose=True)

    # Calculate SNR in source space:
    snr_stc = stc.estimate_snr(group.info, fwd, cov)

    # Plot an average SNR across source points over time:
    ave = np.mean(snr_stc.data, axis=0)

    fig, ax = plt.subplots()
    ax.plot(group.times, ave)
    ax.set(xlabel='Time (sec)', ylabel='SNR EEG', title=names[counter])
    fig.tight_layout()

    # Find time point of maximum SNR
    maxidx = np.argmax(ave)

    # Plot SNR on source space at the time point of maximum SNR:
    kwargs = dict(initial_time=group.times[maxidx], hemi='split',
              views=['lat', 'med'], subjects_dir=subjects_dir, size=(600, 600),
              clim=dict(kind='value', lims=(-100, -70, -40)),
              transparent=True, colormap='viridis')
    brain = snr_stc.plot(**kwargs)
    brain.add_text(0.1, 0.9, f'Condition: {names[counter]}', 'title', font_size=16)

    counter+=1